# P1 — Inventorier les tables, puis les fuites

**Enchaînement : 1.1 Inventorier les tables → 1.8 Fuites**

Phase 1 du projet *Santander Satisfaction* : connaître les données avant de modéliser.
Chaque tâche suit le même plan : **la question**, **la mesure** (le code), **le constat**, **la décision**
que le porteur doit pouvoir défendre seul.

Pour relancer : *Exécuter tout*. Les données sont lues dans `data/` (récupérées par `git pull`), tables entières.

In [1]:
# --- Réglages communs à toute l'équipe ------------------------------------
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", 80)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.4f}".format)

# Les données sont dans data/ à la racine du dépôt (récupérées par git pull)
DATA = next(p for p in (Path("data"), Path("../data")) if (p / "train.csv").is_file())
CIBLE, ID = "TARGET", "ID"
GRAINE = 42             # même graine pour toute l'équipe : même découpage
PART_VALIDATION = 0.20  # protocole : 80 % apprentissage / 20 % validation, stratifié
PLANCHER = 0.6890       # AUC de la règle métier, donnée par l'énoncé
SENTINELLES = (-999999, 9999999999)
FAMILLES = ("saldo", "imp", "num", "ind", "delta")

t0 = time.perf_counter()
train = pd.read_csv(DATA / "train.csv")
a_decider = pd.read_csv(DATA / "a_decider.csv")
DUREE_CHARGEMENT = time.perf_counter() - t0
VARIABLES = [c for c in train.columns if c not in (ID, CIBLE)]
y = train[CIBLE]


def famille(col):
    if col in (ID, CIBLE):
        return col
    debut = col.split("_")[0]
    return debut if debut in FAMILLES else "isolée"


def pct(x):
    return f"{100 * x:.2f} %"


print(f"train : {train.shape} · a_decider : {a_decider.shape} · lus en {DUREE_CHARGEMENT:.1f} s depuis {DATA}")

train : (60816, 371) · a_decider : (15204, 370) · lus en 0.8 s depuis ../data


## 1.1 — Inventorier les tables

**Question** : quelles tables avons-nous, de quelle taille, avec quelles colonnes ? Correspondent-elles à l'énoncé ?

In [2]:
lignes = []
for f in sorted(DATA.glob("*.csv")):
    with open(f, encoding="utf-8") as h:
        entete = h.readline().strip().split(",")
        n = sum(1 for _ in h)
    lignes.append((f.name, round(f.stat().st_size / 1e6, 1), n, len(entete)))
pd.DataFrame(lignes, columns=["fichier", "Mo", "lignes", "colonnes"])

,fichier,Mo,lignes,colonnes
0,a_decider.csv,15.1000,15204,370
1,train.csv,60.5000,60816,371


In [ ]:
print("Seulement dans train     :", sorted(set(train.columns) - set(a_decider.columns)))
print("Seulement dans a_decider :", sorted(set(a_decider.columns) - set(train.columns)) or "aucune")
print("Même ordre de colonnes   :", list(train.columns.drop(CIBLE)) == list(a_decider.columns))

Seulement dans train     : ['TARGET']
Seulement dans a_decider : aucune
Même ordre de colonnes   : True


In [4]:
attendu = {"saldo": 71, "imp": 49, "num": 143, "ind": 75, "delta": 26, "isolée": 6, ID: 1, CIBLE: 1}
familles = pd.DataFrame({"mesuré": pd.Series([famille(c) for c in train.columns]).value_counts(),
                         "énoncé": pd.Series(attendu)}).fillna(0).astype(int)
familles["écart"] = familles["mesuré"] - familles["énoncé"]
display(familles)
print("Variables isolées :", [c for c in train.columns if famille(c) == "isolée"])
print("Total énoncé :", sum(attendu.values()), "colonnes · total réel :", train.shape[1])

,mesuré,énoncé,écart
ID,1,1,0
TARGET,1,1,0
delta,26,26,0
imp,49,49,0
ind,75,75,0
isolée,5,6,-1
num,143,143,0
saldo,71,71,0


Variables isolées : ['var3', 'var15', 'var36', 'var21', 'var38']
Total énoncé : 372 colonnes · total réel : 371


**Constat**

- Deux tables : `train.csv` (60 816 clients × 371 colonnes) et `a_decider.csv` (15 204 × 370). Seule différence : `TARGET`, absente de `a_decider` — normal, c'est ce qu'on doit prédire.
- Les familles correspondent à l'énoncé, **sauf les variables isolées : 5 et non 6** (`var3`, `var15`, `var21`, `var36`, `var38`). Les familles de l'énoncé totalisent 372 colonnes pour 371 : l'énoncé en compte une de trop.

**Décision**

- `train.csv` sert à apprendre **et** à valider (découpage en 1.10). `a_decider.csv` ne sert qu'à produire la décision finale : on ne l'utilise jamais pour choisir un modèle ou un réglage.

## 1.8 — Fuites

**Question** : une variable contient-elle une information **inconnue au moment où le dossier est ouvert** (par exemple, une conséquence de la plainte) ?

**Méthode** : une fuite se trahit par un pouvoir de prédiction trop beau. On mesure l'AUC de **chaque variable prise seule** : une valeur proche de 1 serait suspecte. On vérifie aussi que l'`ID` ne dit rien, et que `a_decider` ressemble à `train`.

In [5]:
from sklearn.metrics import roc_auc_score

def auc_sym(cible, x):
    """AUC d'une variable prise seule comme score, quel que soit son sens."""
    a = roc_auc_score(cible, x)
    return max(a, 1 - a)

non_constantes = [c for c in VARIABLES if train[c].nunique() > 1]
auc = pd.Series({c: auc_sym(y, train[c]) for c in non_constantes}).sort_values(ascending=False)
print(f"{(auc > 0.80).sum()} variable(s) au-dessus de 0,80")
auc.head(15).rename("AUC seule").to_frame()

0 variable(s) au-dessus de 0,80


,AUC seule
var15,0.7010
saldo_var30,0.6952
num_meses_var5_ult3,0.6930
saldo_var42,0.6801
num_var30,0.6795
saldo_var5,0.6785
saldo_medio_var5_hace2,0.6785
saldo_medio_var5_ult1,0.6764
saldo_medio_var5_ult3,0.6752
ind_var30,0.6730


In [6]:
print(f"AUC de l'ID seul : {auc_sym(y, train[ID]):.4f}")
train.groupby(pd.qcut(train[ID], 10), observed=True)[CIBLE].mean().rename("taux de mécontents par dixième d'ID").to_frame()

AUC de l'ID seul : 0.5076


,taux de mécontents par dixième d'ID
ID,
"(0.999, 15363.0]",0.0403
"(15363.0, 30640.0]",0.0393
"(30640.0, 45829.5]",0.0405
"(45829.5, 60945.0]",0.0391
"(60945.0, 76264.5]",0.0331
"(76264.5, 91493.0]",0.0378
"(91493.0, 106279.0]",0.0398
"(106279.0, 121394.0]",0.0385
"(121394.0, 136506.5]",0.0429


In [7]:
# Train contre a_decider : peut-on deviner de quelle table vient un client ?
origine = np.r_[np.zeros(len(train)), np.ones(len(a_decider))]
derive = pd.Series({c: auc_sym(origine, pd.concat([train[c], a_decider[c]])) for c in non_constantes})
derive.sort_values(ascending=False).head(8).rename("AUC train / a_decider").to_frame()

,AUC train / a_decider
var38,0.5028
num_var22_ult1,0.5028
num_meses_var5_ult3,0.5027
saldo_medio_var5_hace3,0.5026
saldo_medio_var5_hace2,0.5024
num_var41_0,0.5020
num_var42_0,0.5019
ind_var41_0,0.5019


**Constat**

- La meilleure variable seule (`var15`) atteint une AUC de **0,70**. **Aucune ne dépasse 0,80** : pas de variable « trop belle ».
- L'`ID` seul : AUC **0,51**, et le taux de mécontents est plat d'un dixième d'ID à l'autre (3,3 % à 4,4 %). L'ordre des dossiers ne trahit rien.
- `train` et `a_decider` sont indiscernables variable par variable (AUC ≤ 0,503) : même population.

**Décision**

- **Aucune variable n'est retirée pour fuite.** Les colonnes `delta_` (variations entre deux dates) restent à surveiller : leurs deux dates doivent précéder la plainte, ce que les données anonymisées ne permettent pas de vérifier.
- **La fuite la plus probable sera la nôtre** : tout traitement appris sur les données (imputation, sélection de variables, choix du seuil) doit être appris **sur l'apprentissage seul**, à l'intérieur du pipeline, jamais sur `train` entier avant découpage.